# IMQCAM File Collections: Builds, Parts, & Data Downloads

In the IMQCAM DMS:
- **Builds (Parent IGSNs)**: e.g. `CMXMAG00001` represents an entire manufacturing build.
- **Parts / Specimens (Child IGSNs)**: e.g. `CMXMAG00001-039` represents individual parts/specimens excised from that build.
- **Linked Metadata**: Material composition (`alloy718`, `Ti-6Al-4V`), AM parameters, and heat treatment schedules are attached to these IGSNs via form entries.
- **Related Data Files**: Measurement files (`ebsd_raw`, `ebsd_derived`, `optical_microscopy`, `ct_tomography`, etc.) are tagged with the part's `igsn` and `data_type`.

This notebook demonstrates how modelers can:
1. **Inspect all Builds & Parts** across the repository.
2. **Filter by Material / Composition** (e.g., `alloy718` / `IN718`, `Ti-6Al-4V`).
3. **Map Data Availability** (which parts have EBSD, microscopy, etc.).
4. **Query & Download Files** for target parts into structured local directories.

In [84]:
import json
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "imqcam.py").exists())
sys.path.insert(0, str(REPO_ROOT))

from imqcam import get_client, fetch_entries, entries_to_frame, extract_igsn

client = get_client()
print(f"Connected to {client.urlBase}")

Connected to https://data.imqcam.org/api/v1/


## 1. Data Types: Total Files Counted vs. Queryable Items

- **`GET /imqcam/count`** counts physical files on the Girder storage server.
- **`GET /imqcam/datafiles`** queries Girder **Items** that carry specimen IGSN metadata.
- **Why they differ**:
  1. **Item vs. File granularity**: A single EBSD acquisition item often bundles multiple files (e.g., data scan + companion header/preview).
  2. **Metadata Readiness**: Files uploaded without complete sample indexing (`meta.igsn`, `experiment_date`) are counted on storage, but are not yet queryable via `/imqcam/datafiles` (e.g. `ct_tomography`).

In [85]:
data_types = [d for d in client.get("imqcam/datatype") if d]
counts = client.get("imqcam/count")

rows = []
for d in data_types:
    files_counted = counts.get(d, 0)
    # Fetch retrievable items count with limit=1 to check availability quickly
    sample_batch = client.get("imqcam/datafiles", parameters={"dataType": d, "limit": 100})
    rows.append({
        "Data Type": d,
        "Files Counted (Server Storage)": files_counted,
        "Status": "Queryable via IGSN" if len(sample_batch) > 0 else "Pending IGSN Tagging"
    })

inventory_df = pd.DataFrame(rows).sort_values("Files Counted (Server Storage)", ascending=False).reset_index(drop=True)
inventory_df

,Data Type,Files Counted (Server Storage),Status
0,ebsd_raw,12824,Queryable via IGSN
1,ct_tomography,7148,Pending IGSN Tagging
2,ebsd_derived,447,Queryable via IGSN
3,fractography_raw,23,Queryable via IGSN


## 2. Fetching & Flattening File Collections

`/imqcam/datafiles` caps page size at 100, so `fetch_datafiles()` pages through offset automatically. `datafiles_to_frame()` unpacks Girder metadata into a tabular DataFrame.

In [86]:
def fetch_datafiles(client, data_type, igsn=None, extra_fields=None, page_size=100):
    """Fetch datafiles, optionally filtered by specimen IGSN."""
    params = {"dataType": data_type, "limit": page_size}
    if igsn and igsn != "Any":
        params["filters"] = json.dumps({"meta.igsn": igsn})
    if extra_fields:
        params["extraFields"] = json.dumps(extra_fields)

    out, offset = [], 0
    while True:
        params["offset"] = offset
        batch = client.get("imqcam/datafiles", parameters=params)
        out.extend(batch)
        if len(batch) < page_size:
            return out
        offset += page_size

In [87]:
def datafiles_to_frame(items):
    """Flatten Girder item envelopes and metadata into one row per item."""
    if not items:
        return pd.DataFrame()

    envelope = pd.DataFrame([
        {key: value for key, value in item.items() if key != "meta"}
        for item in items
    ])
    meta = pd.json_normalize(
        [item.get("meta", {}) for item in items],
        sep=".",
    )
    meta = meta.drop(
        columns=[column for column in meta.columns if column in envelope.columns],
        errors="ignore",
    )
    return pd.concat([envelope, meta], axis=1)

## 3. Inventory: Builds (Parents), Parts (Children), & Compositions

Every specimen cut from an additive build shares a parent IGSN root (e.g. build `CMXMAG00001` has child parts `CMXMAG00001-001`, `CMXMAG00001-039`, etc.).
We pull form entries to map all builds, their parts count, and their material composition (e.g., `alloy718`, `Ti-6Al-4V`).

In [88]:
# Fetch all structured form entries
entries = fetch_entries(client, page_size=100)
entries_df = entries_to_frame(entries)

print(f"Retrieved {len(entries_df)} form entries")

Retrieved 761 form entries


In [89]:
# Keep entries with an IGSN and derive the parent build IGSN
valid_entries = entries_df[entries_df["igsn"].notna()].copy()
valid_entries["build_igsn"] = valid_entries["igsn"].apply(
    lambda value: str(value).split("-")[0] if "-" in str(value) else str(value)
)
valid_entries["is_part"] = valid_entries["igsn"].astype(str).str.contains("-")

print(f"Entries with IGSNs: {len(valid_entries)}")
print(f"Distinct builds: {valid_entries['build_igsn'].nunique()}")
print(f"Distinct parts: {valid_entries.loc[valid_entries['is_part'], 'igsn'].nunique()}")

Entries with IGSNs: 503
Distinct builds: 20
Distinct parts: 223


In [96]:
# Extract material names from the form-entry columns
def get_material(row):
    for column in ["alloy", "powderDetails.alloy", "material", "composition"]:
        if column in row and pd.notna(row[column]):
            return str(row[column])

    for column in ["buildName", "uniqueId"]:
        if column in row and pd.notna(row[column]):
            value = str(row[column])
            if "718" in value:
                return "alloy718"
            if "Ti64" in value or "Ti" in value:
                return "Ti-6Al-4V"

    return "Unspecified"

valid_entries["raw_material"] = valid_entries.apply(get_material, axis=1)
valid_entries[["igsn", "build_igsn", "raw_material"]].drop_duplicates().head(5)

,igsn,build_igsn,raw_material
204,CMXMAL00007,CMXMAL00007,Ti-6Al-4V
205,CMXMAL00008,CMXMAL00008,Ti-6Al-4V
206,CMXMAL00010,CMXMAL00010,Ti-6Al-4V
207,CMXMAL00011-01,CMXMAL00011,Unspecified
208,CMXMAL00011-02,CMXMAL00011,Unspecified


In [97]:
# Propagate material from each parent build to its child parts
build_to_material = (
    valid_entries[valid_entries["raw_material"] != "Unspecified"]
    .groupby("build_igsn")["raw_material"]
    .first()
    .to_dict()
)
valid_entries["material"] = (
    valid_entries["build_igsn"].map(build_to_material).fillna("Unspecified")
)

valid_entries[["igsn", "build_igsn", "material"]].drop_duplicates().head()

,igsn,build_igsn,material
204,CMXMAL00007,CMXMAL00007,Ti-6Al-4V
205,CMXMAL00008,CMXMAL00008,Ti-6Al-4V
206,CMXMAL00010,CMXMAL00010,Ti-6Al-4V
207,CMXMAL00011-01,CMXMAL00011,Unspecified
208,CMXMAL00011-02,CMXMAL00011,Unspecified


In [98]:
# Summarize builds, part counts, materials, and linked forms
build_summary = valid_entries.groupby("build_igsn").agg(
    total_parts=("igsn", lambda values: values[values.astype(str).str.contains("-")].nunique()),
    material=("material", "first"),
    forms_linked=("formName", lambda values: list(values.dropna().unique())),
).reset_index()

print(f"Registered builds: {len(build_summary)}")
build_summary

Registered builds: 20


,build_igsn,total_parts,material,forms_linked
0,APLMAK00002,0,Unspecified,[AM Build Parameters]
1,APLMAK00003,0,Unspecified,[AM Build Parameters]
2,APLMAK00004,0,Unspecified,[AM Build Parameters]
3,APLMAL00001,10,Ti-6Al-4V,"[Printer Build, Sample Heat Treatment Record, ..."
4,CMXMAG00001,10,alloy718,"[Printer Build, AM Build Parameters, Sample He..."
5,CMXMAG00002,3,alloy718,"[Printer Build, AM Build Parameters]"
6,CMXMAK00001,2,Unspecified,[AM Build Parameters]
7,CMXMAL00007,5,Ti-6Al-4V,"[Printer Build, AM Build Parameters, Sample He..."
8,CMXMAL00008,0,Ti-6Al-4V,"[Printer Build, AM Build Parameters]"
9,CMXMAL00010,0,Ti-6Al-4V,"[Printer Build, AM Build Parameters]"


## 4. Query Data Files by Composition & Data Type

Filter by material (e.g., `alloy718`) or a specific build (`CMXMAG00001`), and query the measurement files (`ebsd_raw`, `ebsd_derived`, etc.) available for each part.

In [99]:
# Configure the file query
target_datatype = "ebsd_raw"
target_material = "alloy718"  # Use "" for any material
target_part = "CMXMAG00001-039"                # Use an IGSN such as "CMXMAG00001-039", or "" for all parts

In [100]:
# Fetch matching items
if target_part:
    print(f"Querying '{target_datatype}' for part {target_part}...")
    matched_files = fetch_datafiles(client, target_datatype, igsn=target_part)
elif target_material:
    print(f"Querying '{target_datatype}' for material {target_material}...")
    all_datatype_items = fetch_datafiles(client, target_datatype)
    all_datatype_df = datafiles_to_frame(all_datatype_items)
    material_parts = set(
        valid_entries.loc[
            (valid_entries["material"] == target_material) & valid_entries["is_part"],
            "igsn",
        ]
    )
    matched_ids = set(
        all_datatype_df.loc[all_datatype_df["igsn"].isin(material_parts), "_id"]
    )
    matched_files = [
        item for item in all_datatype_items if item.get("_id") in matched_ids
    ]
else:
    print(f"Querying all '{target_datatype}' items...")
    matched_files = fetch_datafiles(client, target_datatype)

print(f"Retrieved {len(matched_files)} items")

Querying 'ebsd_raw' for part CMXMAG00001-039...
Retrieved 6247 items


In [101]:
# Convert matching items into a DataFrame and inspect coverage
df_files = datafiles_to_frame(matched_files)

part_count = df_files["igsn"].nunique() if "igsn" in df_files.columns else 0
print(f"Found {len(df_files)} '{target_datatype}' items across {part_count} part(s)")

if len(df_files) and "igsn" in df_files.columns:
    print("\nItems per part:")
    print(df_files["igsn"].value_counts().to_string())

Found 6247 'ebsd_raw' items across 1 part(s)

Items per part:
igsn
CMXMAG00001-039    6247


In [102]:
# Preview file metadata
if len(df_files):
    columns_to_show = [
        column for column in ["name", "size", "data_type", "igsn", "created"]
        if column in df_files.columns
    ]
    display(df_files[columns_to_show].head(10))

,name,size,data_type,igsn,created
0,00116eef-595d-44fb-b9a1-0fa7506dd3fb.dat,1966080,ebsd_raw,CMXMAG00001-039,2026-09-10T21:42:11.974000+00:00
1,001b18d7-2a05-40ea-b115-68d85a7bd616.dat,5701650,ebsd_raw,CMXMAG00001-039,2026-09-10T21:39:11.233000+00:00
2,0035fdd6-5afa-435a-b073-8c7db66c56a6.dat,1966080,ebsd_raw,CMXMAG00001-039,2026-09-10T21:42:01.734000+00:00
3,0036210e-eb2c-4909-b839-9abf2497b7ce.dat,1966080,ebsd_raw,CMXMAG00001-039,2026-09-10T21:42:42.343000+00:00
4,00474464-dd88-4291-a6e7-b058bf5773d4.dat,1966080,ebsd_raw,CMXMAG00001-039,2026-09-10T21:42:57.776000+00:00
5,005cd894-b18b-4400-bbbe-2751f186412c.dat,196608,ebsd_raw,CMXMAG00001-039,2026-09-10T21:40:10.929000+00:00
6,00864ff4-bac0-4984-bb3f-a2307cd4800f.dat,786432,ebsd_raw,CMXMAG00001-039,2026-09-10T21:42:21.844000+00:00
7,009283dd-44eb-41be-a697-02d47a55605a.dat,1966080,ebsd_raw,CMXMAG00001-039,2026-09-10T21:42:37.849000+00:00
8,00aeb2f6-5826-4fb6-86ba-b34a55bb52db.dat,786432,ebsd_raw,CMXMAG00001-039,2026-09-10T21:37:27.610000+00:00
9,00b82d9e-16be-4bf4-bc3b-3f861ba80727.dat,1966080,ebsd_raw,CMXMAG00001-039,2026-09-10T21:37:36.928000+00:00


## 5. Download Files Locally to Structured Directories

Save the measurement files organized by build and part folder:
`downloads/<material>/<build_igsn>/<part_igsn>/<filename>`
This structure allows external modelers, simulation meshes, and ML pipelines to locate input files by specimen ID without manual folder creation.

In [103]:
# Configure a download preview
# Keep this False while testing; set True only after checking the result table.
download_files = True
download_limit = 3  # Use None to download every matched item

In [104]:
# Download the matched items
if download_files:
    items_to_download = matched_files
    if download_limit is not None:
        items_to_download = items_to_download[:download_limit]

    for item in items_to_download:
        item_igsn = item.get("meta", {}).get("igsn", target_part or "unknown_part")
        build_igsn = item_igsn.split("-")[0] if "-" in item_igsn else "general"
        material_folder = target_material or "general"
        destination = (
            REPO_ROOT
            / "downloads"
            / target_datatype
            / material_folder
            / build_igsn
            / item_igsn
        )
        destination.mkdir(parents=True, exist_ok=True)
        client.downloadItem(item["_id"], dest=str(destination))

    print(f"Downloaded {len(items_to_download)} item(s)")
else:
    print("Downloads are disabled. Set download_files = True to enable them.")

Downloaded 3 item(s)


## 6. Requesting Custom Metadata with `extraFields` (e.g. `meta.composition`)

By default, `/imqcam/datafiles` projects only default envelope fields (`data_type`, `igsn`) for speed.
To retrieve custom metadata keys attached to items (such as `meta.composition = "IN718"`), pass `extraFields=json.dumps(["meta.composition"])`.
Once loaded into the DataFrame, the `composition` column is immediately available for filtering in pandas.

In [105]:
# Fetch one specimen with custom item metadata
specimen_igsn = "CMXMAG00001-042"
raw_items = fetch_datafiles(
    client,
    data_type="ebsd_raw",
    igsn=specimen_igsn,
    extra_fields=["meta.composition"],
)
specimen_df = datafiles_to_frame(raw_items)

print(f"Retrieved {len(specimen_df)} items for {specimen_igsn}")
print("Available columns:", list(specimen_df.columns))

Retrieved 6471 items for CMXMAG00001-042
Available columns: ['_id', '_modelType', 'baseParentId', 'baseParentType', 'created', 'creatorId', 'folderId', 'name', 'size', 'composition', 'data_type', 'igsn']


In [106]:
# Filter the DataFrame by composition
target_comp = "718"
if "composition" in specimen_df.columns:
    comp_df = specimen_df[
        specimen_df["composition"].astype(str).str.contains(
            target_comp,
            case=False,
            na=False,
        )
    ]
    print(f"Found {len(comp_df)} items with composition matching '{target_comp}'")
    display(comp_df[["name", "size", "data_type", "igsn", "composition"]].head())
else:
    print("Composition metadata was not returned.")
    display(specimen_df[["name", "size", "data_type", "igsn"]].head())

Found 6471 items with composition matching '718'


,name,size,data_type,igsn,composition
0,0002b8d8-b3d5-4cdd-ad5c-beb469af1cda.dat,1966080,ebsd_raw,CMXMAG00001-042,IN718
1,00036d0c-427a-4ccc-be1a-5191c366fb1c.dat,1966080,ebsd_raw,CMXMAG00001-042,IN718
2,00084f92-6d03-4609-99c6-cdae26f3b314.dat,196608,ebsd_raw,CMXMAG00001-042,IN718
3,00160122-4f09-4904-8e30-79374097f859.dat,786432,ebsd_raw,CMXMAG00001-042,IN718
4,001e1abb-4b0b-4c78-a852-a90715d1bc04.dat,1966080,ebsd_raw,CMXMAG00001-042,IN718


In [107]:
# Download the composition-filtered files
# Keep this False while testing; set it to True for the complete filtered collection.
download_filtered_files = True
download_limit = 3  # Set to an integer such as 3 for a small test

if download_filtered_files:
    filtered_ids = set(comp_df["_id"])
    filtered_items = [
        item for item in raw_items if item.get("_id") in filtered_ids
    ]
    if download_limit is not None:
        filtered_items = filtered_items[:download_limit]

    download_dir = (
        REPO_ROOT
        / "downloads"
        / "ebsd_raw"
        / specimen_igsn
        / target_comp
    )
    download_dir.mkdir(parents=True, exist_ok=True)

    print(f"Downloading {len(filtered_items)} filtered item(s) to {download_dir}...")
    for item in filtered_items:
        client.downloadItem(item["_id"], dest=str(download_dir))
    print(f"✅ Download complete: {download_dir}")
else:
    print("Downloads are disabled. Set download_filtered_files = True to enable them.")

✅ Download complete: /Users/akhilaponugoti/Documents/imqcam_dms_api/downloads/ebsd_raw/CMXMAG00001-042/718
